# Sistema de Regulação de Sinistros Paramétricos

## Versão BETA Final - Kovr Seguradora

Este notebook permite processar regulações de sinistros paramétricos de forma standalone,
sem necessidade de servidor ou infraestrutura externa.

### Fontes de Dados
- **Precipitação**: CHIRPS (Google Earth Engine)
- **Temperatura**: AgERA5 (CDS Copernicus)

### Lógica de Cálculo
- **Precipitação**: Soma total do período. Sinistro se total < strike.
- **Temperatura**: Valor mínimo do período. Sinistro se mínimo < strike.
- **Franquia**: Deduzida do valor bruto. Se indenização bruta ≤ franquia, pagamento = R$ 0,00.

---

## 1. Instalação de Dependências

Execute esta célula apenas uma vez para instalar as bibliotecas necessárias.

In [ ]:
# Instalar dependências (execute apenas uma vez)
!pip install earthengine-api cdsapi netCDF4 xarray pandas openpyxl --quiet
print("Dependências instaladas com sucesso!")

## 2. Configuração de Credenciais

### 2.1 Google Earth Engine (para Precipitação)

Se você ainda não tem uma conta no Earth Engine:
1. Acesse https://earthengine.google.com/
2. Clique em "Sign Up"
3. Siga as instruções para criar uma conta

In [ ]:
# Autenticação do Earth Engine
import ee

try:
    ee.Initialize()
    print("✅ Earth Engine já está autenticado!")
except:
    print("Iniciando autenticação do Earth Engine...")
    ee.Authenticate()
    ee.Initialize()
    print("✅ Earth Engine autenticado com sucesso!")

### 2.2 CDS Copernicus (para Temperatura)

Se você ainda não tem uma conta no CDS:
1. Acesse https://cds.climate.copernicus.eu/
2. Clique em "Register"
3. Após criar a conta, vá em Profile > API Key
4. Copie seu UID e API Key

In [ ]:
# Configuração do CDS API
# Substitua pelos seus valores
CDS_UID = "SEU_UID_AQUI"  # Ex: "123456"
CDS_API_KEY = "SUA_API_KEY_AQUI"  # Ex: "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"

# Criar arquivo de configuração
import os
cdsapirc_path = os.path.expanduser("~/.cdsapirc")

if CDS_UID != "SEU_UID_AQUI" and CDS_API_KEY != "SUA_API_KEY_AQUI":
    with open(cdsapirc_path, 'w') as f:
        f.write(f"url: https://cds.climate.copernicus.eu/api\n")
        f.write(f"key: {CDS_UID}:{CDS_API_KEY}\n")
    print(f"✅ Arquivo de configuração CDS criado em: {cdsapirc_path}")
else:
    if os.path.exists(cdsapirc_path):
        print("✅ Arquivo de configuração CDS já existe!")
    else:
        print("⚠️ Por favor, insira seu UID e API Key do CDS nas variáveis acima.")

## 3. Funções do Sistema

Execute esta célula para carregar todas as funções necessárias.

In [ ]:
import re
import os
import tempfile
import zipfile
import glob
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import ee

def parse_html_content(html_content):
    """Extrai informações da apólice do conteúdo HTML."""
    result = {}
    
    # Detectar tipo de cobertura
    if 'CHIRPS' in html_content.upper():
        result['data_provider'] = 'CHIRPS'
        result['type_of_cover'] = 'precipitation'
    elif 'AGERA5' in html_content.upper() or 'ERA5' in html_content.upper():
        result['data_provider'] = 'AgERA5'
        result['type_of_cover'] = 'temperature'
    else:
        if 'precipit' in html_content.lower() or 'chuva' in html_content.lower():
            result['data_provider'] = 'CHIRPS'
            result['type_of_cover'] = 'precipitation'
        elif 'temperat' in html_content.lower() or 'frio' in html_content.lower():
            result['data_provider'] = 'AgERA5'
            result['type_of_cover'] = 'temperature'
        else:
            result['data_provider'] = 'Unknown'
            result['type_of_cover'] = 'unknown'
    
    # Extrair período
    period_pattern = r'Period\s*[Cc]over[:\s]*(\d{4}-\d{2}-\d{2})\s*(?:to|até|a|-)\s*(\d{4}-\d{2}-\d{2})'
    period_match = re.search(period_pattern, html_content, re.IGNORECASE)
    if period_match:
        result['period_start'] = period_match.group(1)
        result['period_end'] = period_match.group(2)
    else:
        date_pattern = r'(\d{4}-\d{2}-\d{2})'
        dates = re.findall(date_pattern, html_content)
        if len(dates) >= 2:
            result['period_start'] = dates[0]
            result['period_end'] = dates[1]
    
    # Extrair coordenadas do Leaflet
    leaflet_pattern = r'setView\(\s*\[\s*([-\d.]+)\s*,\s*([-\d.]+)\s*\]'
    leaflet_match = re.search(leaflet_pattern, html_content)
    if leaflet_match:
        result['latitude'] = float(leaflet_match.group(1))
        result['longitude'] = float(leaflet_match.group(2))
    
    # Extrair Strike - padrões para precipitação e temperatura
    # Padrão para temperatura: "Strike temperature : 3 °C" ou "Strike temperature : -10 °C"
    strike_temp_match = re.search(r'[Ss]trike\s+[Tt]emperature\s*[:\s]*(-?[\d.]+)', html_content)
    if strike_temp_match:
        result['strike'] = float(strike_temp_match.group(1))
    else:
        # Padrão genérico para Strike
        strike_match = re.search(r'[Ss]trike[:\s]*([\d.]+)', html_content)
        if strike_match:
            result['strike'] = float(strike_match.group(1))
    
    # Extrair Exit Point - padrões para precipitação e temperatura
    # Padrão para temperatura: "Exit temperature : -10 °C"
    exit_temp_match = re.search(r'[Ee]xit\s+[Tt]emperature\s*[:\s]*(-?[\d.]+)', html_content)
    if exit_temp_match:
        result['exit_point'] = float(exit_temp_match.group(1))
    else:
        # Padrão genérico para Exit Point
        exit_match = re.search(r'[Ee]xit\s*[Pp]oint[:\s]*([\d.]+)', html_content)
        if exit_match:
            result['exit_point'] = float(exit_match.group(1))
    
    # Extrair Limit of Indemnity
    limit_match = re.search(r'[Ll]imit[:\s]*([\d,.]+)', html_content)
    if limit_match:
        result['limit'] = float(limit_match.group(1).replace(',', ''))
    
    # Extrair Tick
    tick_match = re.search(r'[Tt]ick[:\s]*([\d,.]+)', html_content)
    if tick_match:
        result['tick'] = float(tick_match.group(1).replace(',', ''))
    
    # Extrair Deductible (franquia)
    deductible_match = re.search(r'[Dd]eductible[:\s]*([\d.]+)\s*%?', html_content)
    if deductible_match:
        result['deductible'] = float(deductible_match.group(1))
    else:
        result['deductible'] = 0
    
    return result

def fetch_chirps_data(latitude, longitude, start_date, end_date):
    """Busca dados de precipitação do CHIRPS via Google Earth Engine."""
    point = ee.Geometry.Point([longitude, latitude])
    
    # Ajustar end_date para incluir o último dia
    end_dt = datetime.strptime(end_date, '%Y-%m-%d') + timedelta(days=1)
    end_date_adjusted = end_dt.strftime('%Y-%m-%d')
    
    collection = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
        .filterDate(start_date, end_date_adjusted) \
        .filterBounds(point)
    
    def extract_value(image):
        value = image.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=5566
        ).get('precipitation')
        return ee.Feature(None, {
            'date': image.date().format('YYYY-MM-DD'),
            'value': value
        })
    
    features = collection.map(extract_value)
    result = features.getInfo()
    
    data = []
    for feature in result['features']:
        props = feature['properties']
        data.append({
            'date': props['date'],
            'value': props['value'] if props['value'] is not None else 0
        })
    
    data.sort(key=lambda x: x['date'])
    return data

def fetch_agera5_data(latitude, longitude, start_date, end_date, statistic='24_hour_minimum'):
    """Busca dados de temperatura do AgERA5 via CDS Copernicus API.
    
    IMPORTANTE: O CDS pode retornar um arquivo ZIP com múltiplos arquivos NetCDF
    (um por mês). Esta função processa todos os arquivos corretamente.
    """
    import cdsapi
    import xarray as xr
    
    # Converter coordenadas para bounding box (±0.1°)
    north = latitude + 0.1
    south = latitude - 0.1
    west = longitude - 0.1
    east = longitude + 0.1
    
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')
    
    years = list(set([str(y) for y in range(start_dt.year, end_dt.year + 1)]))
    months = [f"{m:02d}" for m in range(1, 13)]
    days = [f"{d:02d}" for d in range(1, 32)]
    
    client = cdsapi.Client()
    
    # Criar diretório temporário para processar arquivos
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = os.path.join(tmp_dir, 'download.zip')
        
        request = {
            'variable': '2m_temperature',
            'statistic': [statistic],
            'year': years,
            'month': months,
            'day': days,
            'version': '2_0',
            'area': [north, west, south, east]
        }
        
        print(f"Baixando dados do CDS Copernicus...")
        print(f"Período: {start_date} a {end_date}")
        print(f"Coordenadas: lat={latitude}, lon={longitude}")
        print("Isso pode levar alguns minutos...")
        
        client.retrieve('sis-agrometeorological-indicators', request, tmp_path)
        
        # Verificar se é um arquivo ZIP
        nc_files = []
        if zipfile.is_zipfile(tmp_path):
            print("Arquivo ZIP detectado, extraindo...")
            with zipfile.ZipFile(tmp_path, 'r') as zip_ref:
                zip_ref.extractall(tmp_dir)
            nc_files = glob.glob(os.path.join(tmp_dir, '*.nc'))
            print(f"Encontrados {len(nc_files)} arquivos NetCDF")
        else:
            # Renomear para .nc se não for ZIP
            nc_path = tmp_path.replace('.zip', '.nc')
            os.rename(tmp_path, nc_path)
            nc_files = [nc_path]
        
        if not nc_files:
            raise ValueError("Nenhum arquivo NetCDF encontrado")
        
        # Processar todos os arquivos NetCDF
        all_data = []
        
        for nc_file in nc_files:
            print(f"Processando: {os.path.basename(nc_file)}")
            ds = xr.open_dataset(nc_file)
            
            # Encontrar variável de temperatura
            temp_var = None
            for var in ds.data_vars:
                if 'temperature' in var.lower() or 't2m' in var.lower():
                    temp_var = var
                    break
            
            if temp_var is None:
                temp_var = list(ds.data_vars)[0]
            
            # Extrair dados
            for time_idx in range(len(ds.time)):
                time_val = pd.Timestamp(ds.time.values[time_idx])
                date_str = time_val.strftime('%Y-%m-%d')
                
                if date_str < start_date or date_str > end_date:
                    continue
                
                temp_data = ds[temp_var].isel(time=time_idx)
                
                if 'lat' in ds.coords:
                    lat_idx = abs(ds.lat - latitude).argmin().item()
                    lon_idx = abs(ds.lon - longitude).argmin().item()
                    value = float(temp_data.isel(lat=lat_idx, lon=lon_idx).values)
                else:
                    value = float(temp_data.mean().values)
                
                # Converter de Kelvin para Celsius se necessário
                if value > 100:
                    value = value - 273.15
                
                all_data.append({
                    'date': date_str,
                    'value': round(value, 2)
                })
            
            ds.close()
        
        # Remover duplicatas e ordenar
        seen_dates = set()
        unique_data = []
        for d in all_data:
            if d['date'] not in seen_dates:
                seen_dates.add(d['date'])
                unique_data.append(d)
        
        unique_data.sort(key=lambda x: x['date'])
        print(f"✅ {len(unique_data)} dias de dados processados")
        
        return unique_data

def calculate_claim(data, params):
    """Calcula o valor do sinistro com aplicação correta da franquia.
    
    Lógica da Franquia:
    1. Calcular indenização bruta (diferença × tick)
    2. Calcular franquia (LMI × deductible%)
    3. Se indenização bruta ≤ franquia → Indenização = R$ 0,00
    4. Se indenização bruta > franquia → Indenização = indenização bruta - franquia
    """
    type_of_cover = params.get('type_of_cover', 'precipitation')
    strike = params.get('strike', 0)
    exit_point = params.get('exit_point', 0)
    limit = params.get('limit', 0)
    tick = params.get('tick', 0)
    deductible_pct = params.get('deductible', 0)  # Em percentual
    
    # Calcular franquia em valor absoluto
    franquia = limit * (deductible_pct / 100) if deductible_pct > 0 else 0
    
    if type_of_cover == 'precipitation':
        # Para precipitação: soma total do período
        observed_value = sum(d['value'] for d in data)
        triggered = observed_value < strike
        if triggered:
            deficit = strike - observed_value
            gross_payout = min(deficit * tick, limit)  # Indenização bruta
        else:
            gross_payout = 0
        value_label = "Precipitação Total"
        unit = "mm"
    else:
        # Para temperatura: valor mínimo do período
        observed_value = min(d['value'] for d in data) if data else 0
        triggered = observed_value < strike
        if triggered:
            deficit = strike - observed_value
            gross_payout = min(deficit * tick, limit)  # Indenização bruta
        else:
            gross_payout = 0
        value_label = "Temperatura Mínima"
        unit = "°C"
    
    # Aplicar franquia corretamente
    if gross_payout <= franquia:
        net_payout = 0  # Franquia absorve toda a indenização
    else:
        net_payout = gross_payout - franquia  # Deduzir franquia
    
    return {
        'observed_value': observed_value,
        'value_label': value_label,
        'unit': unit,
        'strike': strike,
        'deficit': strike - observed_value if triggered else 0,
        'triggered': triggered,
        'gross_payout': gross_payout,
        'franquia': franquia,
        'net_payout': net_payout,
        'daily_data': data
    }

print("✅ Funções carregadas com sucesso!")

## 4. Processamento de Regulação

Cole o conteúdo HTML da apólice na célula abaixo e execute.

In [ ]:
# Cole o conteúdo HTML da apólice aqui
HTML_CONTENT = """
COLE O CONTEÚDO HTML AQUI
"""

# Ou carregue de um arquivo:
# with open('caminho/para/arquivo.html', 'r', encoding='utf-8') as f:
#     HTML_CONTENT = f.read()

In [ ]:
# Extrair parâmetros da apólice
params = parse_html_content(HTML_CONTENT)

print("=" * 60)
print("PARÂMETROS EXTRAÍDOS")
print("=" * 60)
for key, value in params.items():
    print(f"{key}: {value}")

In [ ]:
# Parâmetros de cálculo (preencha se não foram extraídos do HTML)
# Descomente e ajuste os valores conforme necessário

# params['limit'] = 65000  # Limit of Indemnity (R$)
# params['tick'] = 5000    # Tick (R$ por mm ou °C)
# params['deductible'] = 20  # Franquia em percentual (%)
# params['strike'] = 3     # Strike (mm ou °C)

print("Parâmetros de cálculo:")
print(f"  Limit of Indemnity: R$ {params.get('limit', 'N/A'):,.2f}" if params.get('limit') else "  Limit of Indemnity: N/A")
print(f"  Tick: R$ {params.get('tick', 'N/A'):,.2f}" if params.get('tick') else "  Tick: N/A")
print(f"  Deductible: {params.get('deductible', 0)}%")
print(f"  Strike: {params.get('strike', 'N/A')}")

In [ ]:
# Buscar dados climáticos
print("Buscando dados climáticos...")
print(f"Tipo: {params.get('type_of_cover', 'N/A')}")
print(f"Fonte: {params.get('data_provider', 'N/A')}")

if params.get('type_of_cover') == 'precipitation':
    climate_data = fetch_chirps_data(
        params['latitude'],
        params['longitude'],
        params['period_start'],
        params['period_end']
    )
else:
    climate_data = fetch_agera5_data(
        params['latitude'],
        params['longitude'],
        params['period_start'],
        params['period_end']
    )

print(f"\n✅ {len(climate_data)} dias de dados obtidos!")

In [ ]:
# Visualizar dados
df = pd.DataFrame(climate_data)
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

# Definir label baseado no tipo
value_label = "Precipitação (mm)" if params.get('type_of_cover') == 'precipitation' else "Temperatura Mínima (°C)"
df = df.rename(columns={'value': value_label})

print("Primeiros 10 registros:")
display(df.head(10))

print(f"\nÚltimos 10 registros:")
display(df.tail(10))

print(f"\nEstatísticas:")
display(df.describe())

In [ ]:
# Calcular sinistro
claim = calculate_claim(climate_data, params)

print("=" * 70)
print("RESULTADO DO SINISTRO")
print("=" * 70)
print(f"{claim['value_label']} Observada: {claim['observed_value']:.2f} {claim['unit']}")
print(f"Strike (Trigger): {claim['strike']} {claim['unit']}")
print(f"Diferença: {claim['deficit']:.2f} {claim['unit']}")
print(f"")
print(f"Sinistro Acionado: {'SIM ⚠️' if claim['triggered'] else 'NÃO ✅'}")
print(f"")
print(f"Indenização Bruta: R$ {claim['gross_payout']:,.2f}")
print(f"Franquia ({params.get('deductible', 0)}% do LMI): R$ {claim['franquia']:,.2f}")
print(f"")
if claim['gross_payout'] > 0 and claim['gross_payout'] <= claim['franquia']:
    print(f"⚠️ Indenização bruta (R$ {claim['gross_payout']:,.2f}) é menor ou igual à franquia (R$ {claim['franquia']:,.2f})")
    print(f"   Portanto, não há pagamento de indenização.")
print(f"")
print(f"INDENIZAÇÃO FINAL: R$ {claim['net_payout']:,.2f}")
print("=" * 70)

In [ ]:
# Gerar gráfico
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))

# Preparar dados para o gráfico
df_plot = pd.DataFrame(climate_data)
df_plot['date'] = pd.to_datetime(df_plot['date'])

if params.get('type_of_cover') == 'precipitation':
    ax.bar(df_plot['date'], df_plot['value'], color='steelblue', alpha=0.7, label='Precipitação Diária')
    ax.set_ylabel('Precipitação (mm)')
    title = 'Dados de Precipitação - CHIRPS'
else:
    ax.plot(df_plot['date'], df_plot['value'], color='orangered', marker='o', markersize=3, label='Temperatura Mínima')
    ax.fill_between(df_plot['date'], df_plot['value'], alpha=0.3, color='orangered')
    ax.set_ylabel('Temperatura (°C)')
    title = 'Dados de Temperatura Mínima - AgERA5'

if 'strike' in params:
    ax.axhline(y=params['strike'], color='red', linestyle='--', linewidth=2, label=f"Strike: {params['strike']}")

ax.set_xlabel('Data')
ax.set_title(title)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Exportar para Excel
output_file = "regulacao_sinistro.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Aba de dados
    df_export = pd.DataFrame(climate_data)
    df_export.columns = ['Data', claim['value_label'] + f" ({claim['unit']})"]
    df_export.to_excel(writer, sheet_name='Dados Climáticos', index=False)
    
    # Aba de resumo
    summary = pd.DataFrame([
        {'Parâmetro': 'Tipo de Cobertura', 'Valor': params.get('type_of_cover', '').replace('precipitation', 'Precipitação').replace('temperature', 'Temperatura Mínima')},
        {'Parâmetro': 'Fonte de Dados', 'Valor': params.get('data_provider', '')},
        {'Parâmetro': 'Período Início', 'Valor': params.get('period_start', '')},
        {'Parâmetro': 'Período Fim', 'Valor': params.get('period_end', '')},
        {'Parâmetro': 'Latitude', 'Valor': params.get('latitude', '')},
        {'Parâmetro': 'Longitude', 'Valor': params.get('longitude', '')},
        {'Parâmetro': 'Strike', 'Valor': f"{params.get('strike', '')} {claim['unit']}"},
        {'Parâmetro': 'Exit Point', 'Valor': f"{params.get('exit_point', '')} {claim['unit']}" if params.get('exit_point') else ''},
        {'Parâmetro': 'Limit of Indemnity (LMI)', 'Valor': f"R$ {params.get('limit', 0):,.2f}"},
        {'Parâmetro': 'Tick', 'Valor': f"R$ {params.get('tick', 0):,.2f} por {claim['unit']}"},
        {'Parâmetro': 'Deductible (Franquia %)', 'Valor': f"{params.get('deductible', 0)}%"},
        {'Parâmetro': '', 'Valor': ''},
        {'Parâmetro': '--- RESULTADO ---', 'Valor': ''},
        {'Parâmetro': f"{claim['value_label']} Observada", 'Valor': f"{claim['observed_value']:.2f} {claim['unit']}"},
        {'Parâmetro': 'Diferença', 'Valor': f"{claim['deficit']:.2f} {claim['unit']}"},
        {'Parâmetro': 'Sinistro Acionado', 'Valor': 'SIM' if claim['triggered'] else 'NÃO'},
        {'Parâmetro': 'Indenização Bruta', 'Valor': f"R$ {claim['gross_payout']:,.2f}"},
        {'Parâmetro': 'Franquia (Valor)', 'Valor': f"R$ {claim['franquia']:,.2f}"},
        {'Parâmetro': 'INDENIZAÇÃO FINAL', 'Valor': f"R$ {claim['net_payout']:,.2f}"},
    ])
    summary.to_excel(writer, sheet_name='Resumo', index=False)

print(f"✅ Relatório exportado para: {output_file}")

---

## Fim do Processamento

O relatório Excel foi gerado com sucesso. Você pode encontrá-lo no mesmo diretório deste notebook.

### Resumo das Funcionalidades:
- ✅ Extração automática de parâmetros do HTML da apólice
- ✅ Suporte a precipitação (CHIRPS) e temperatura (AgERA5)
- ✅ Processamento de múltiplos arquivos NetCDF (ZIP do CDS)
- ✅ Cálculo correto da franquia sobre o LMI
- ✅ Exportação para Excel com dados e resumo